# 04 — Clusterização de Aeroportos

Objetivo: agrupar aeroportos com **perfis operacionais semelhantes** em relação a atrasos.  
Isso responde diretamente à pergunta do enunciado: *"É possível agrupar aeroportos com perfis semelhantes?"*

Abordagem:
1. Construir features agregadas por aeroporto de origem
2. Determinar o número ideal de clusters com Elbow + Silhouette
3. Rodar KMeans e interpretar os grupos
4. Reduzir para 2D com PCA e visualizar os clusters

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples

RANDOM_STATE = 42
plt.rcParams["figure.dpi"] = 110

## 1. Carregamento e agregação por aeroporto

In [ ]:
df = pl.read_parquet("../data/processed/flights_model.parquet")
print(df.shape)
df.head(3)

In [ ]:
# Agregações por aeroporto de origem
# Usamos apenas aeroportos com volume mínimo para evitar ruído estatístico
MIN_VOOS = 500

airport_features = (
    df
    .filter(pl.col("ARRIVAL_DELAY").is_not_null())
    .group_by("ORIGIN_AIRPORT")
    .agg([
        pl.len().alias("qtd_voos"),

        # Atraso na chegada
        pl.col("ARRIVAL_DELAY").mean().alias("media_arrival_delay"),
        pl.col("ARRIVAL_DELAY").median().alias("mediana_arrival_delay"),
        pl.col("ARRIVAL_DELAY").std().alias("std_arrival_delay"),
        (pl.col("ARRIVAL_DELAY") > 0).mean().alias("taxa_atraso"),
        (pl.col("ARRIVAL_DELAY") > 15).mean().alias("taxa_atraso_grave"),  # >15min

        # Atraso na partida
        pl.col("DEPARTURE_DELAY").mean().alias("media_departure_delay"),

        # Características operacionais
        pl.col("DISTANCE").mean().alias("distancia_media"),
        pl.col("SCHEDULED_TIME").mean().alias("tempo_voo_medio"),
        pl.col("AIRLINE").n_unique().alias("n_airlines"),
        pl.col("DESTINATION_AIRPORT").n_unique().alias("n_destinos"),

        # Mix de período do dia
        (pl.col("periodo_dia") == "manha").mean().alias("pct_manha"),
        (pl.col("periodo_dia") == "tarde").mean().alias("pct_tarde"),
        (pl.col("periodo_dia") == "noite").mean().alias("pct_noite"),
    ])
    .filter(pl.col("qtd_voos") >= MIN_VOOS)
    .sort("qtd_voos", descending=True)
)

print(f"Aeroportos com >= {MIN_VOOS} voos: {airport_features.height}")
airport_features.head(5)

## 2. Pré-processamento das features

In [ ]:
feature_cols = [
    "media_arrival_delay",
    "mediana_arrival_delay",
    "std_arrival_delay",
    "taxa_atraso",
    "taxa_atraso_grave",
    "media_departure_delay",
    "distancia_media",
    "tempo_voo_medio",
    "n_airlines",
    "n_destinos",
    "pct_manha",
    "pct_tarde",
    "pct_noite",
]

ap_pd = airport_features.to_pandas().set_index("ORIGIN_AIRPORT")
X = ap_pd[feature_cols].copy()

# Checar nulos
print("Nulos por feature:")
print(X.isnull().sum())

# Preencher eventuais nulos com mediana
X = X.fillna(X.median())

# Escalar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=feature_cols)

print(f"\nShape para clusterização: {X_scaled.shape}")

## 3. Determinação do número de clusters

In [ ]:
K_RANGE = range(2, 11)
inertias = []
silhouettes = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init="auto")
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))
    print(f"k={k}  inertia={km.inertia_:.1f}  silhouette={silhouettes[-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Elbow
axes[0].plot(list(K_RANGE), inertias, marker="o", color="steelblue")
axes[0].set_title("Método Elbow — Inércia por k")
axes[0].set_xlabel("Número de clusters (k)")
axes[0].set_ylabel("Inércia")
axes[0].grid(alpha=0.3)

# Silhouette
axes[1].plot(list(K_RANGE), silhouettes, marker="o", color="darkorange")
axes[1].set_title("Silhouette Score por k")
axes[1].set_xlabel("Número de clusters (k)")
axes[1].set_ylabel("Silhouette Score")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

best_k = list(K_RANGE)[silhouettes.index(max(silhouettes))]
print(f"Melhor k pelo Silhouette: {best_k}")

## 4. KMeans com k escolhido

In [ ]:
# Ajuste manual se o Silhouette não apontar um k intuitivo
K_FINAL = best_k  # ou substitua por um valor que faça sentido operacional

km_final = KMeans(n_clusters=K_FINAL, random_state=RANDOM_STATE, n_init="auto")
ap_pd["cluster"] = km_final.fit_predict(X_scaled)

print(f"Distribuição de aeroportos por cluster (k={K_FINAL}):")
print(ap_pd["cluster"].value_counts().sort_index())

In [ ]:
# Perfil médio de cada cluster (valores originais, não escalados)
cols_resumo = [
    "taxa_atraso", "taxa_atraso_grave", "media_arrival_delay",
    "media_departure_delay", "std_arrival_delay",
    "distancia_media", "n_destinos", "qtd_voos",
]

perfil = (
    ap_pd[cols_resumo + ["cluster"]]
    .groupby("cluster")
    .mean()
    .round(2)
    .sort_index()
)

perfil

In [ ]:
# Heatmap dos centróides (valores escalados)
centroids_df = pd.DataFrame(
    km_final.cluster_centers_,
    columns=feature_cols,
    index=[f"Cluster {i}" for i in range(K_FINAL)]
)

plt.figure(figsize=(14, max(3, K_FINAL * 0.8)))
sns.heatmap(
    centroids_df,
    annot=True, fmt=".2f", cmap="RdYlGn_r",
    center=0, linewidths=0.5
)
plt.title("Centróides dos clusters (valores padronizados)")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Top aeroportos por volume em cada cluster
for c in sorted(ap_pd["cluster"].unique()):
    subset = ap_pd[ap_pd["cluster"] == c].sort_values("qtd_voos", ascending=False)
    top = subset.head(8).index.tolist()
    taxa = subset["taxa_atraso"].mean()
    print(f"Cluster {c} ({len(subset)} aeroportos | taxa_atraso média={taxa:.2%}): {top}")

## 5. PCA — Redução de dimensionalidade e visualização

In [ ]:
pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_scaled)

variancia_acumulada = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(9, 4))
plt.plot(range(1, len(variancia_acumulada) + 1), variancia_acumulada, marker="o", color="steelblue")
plt.axhline(0.90, color="red", linestyle="--", label="90% variância")
plt.axhline(0.80, color="orange", linestyle="--", label="80% variância")
plt.xlabel("Número de componentes")
plt.ylabel("Variância explicada acumulada")
plt.title("PCA — Variância explicada acumulada")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

n_90 = np.argmax(variancia_acumulada >= 0.90) + 1
print(f"Componentes necessários para 90% da variância: {n_90}")

In [ ]:
# Projeção 2D para visualização dos clusters
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d = pca_2d.fit_transform(X_scaled)

var_pc1 = pca_2d.explained_variance_ratio_[0] * 100
var_pc2 = pca_2d.explained_variance_ratio_[1] * 100

ap_pd["PC1"] = X_2d[:, 0]
ap_pd["PC2"] = X_2d[:, 1]

print(f"PC1 explica {var_pc1:.1f}% | PC2 explica {var_pc2:.1f}% | Total: {var_pc1+var_pc2:.1f}%")

In [ ]:
colors = plt.cm.tab10(np.linspace(0, 1, K_FINAL))

fig, ax = plt.subplots(figsize=(11, 7))

for c, color in zip(sorted(ap_pd["cluster"].unique()), colors):
    mask = ap_pd["cluster"] == c
    subset = ap_pd[mask]
    ax.scatter(
        subset["PC1"], subset["PC2"],
        label=f"Cluster {c} (n={mask.sum()})",
        color=color, s=60, alpha=0.75, edgecolors="white", linewidths=0.4
    )

# Anotar os maiores aeroportos
top_airports = ap_pd.sort_values("qtd_voos", ascending=False).head(20).index
for ap in top_airports:
    row = ap_pd.loc[ap]
    ax.annotate(
        ap, (row["PC1"], row["PC2"]),
        fontsize=7, alpha=0.85,
        xytext=(3, 3), textcoords="offset points"
    )

ax.set_xlabel(f"PC1 ({var_pc1:.1f}% variância)")
ax.set_ylabel(f"PC2 ({var_pc2:.1f}% variância)")
ax.set_title(f"Clusters de aeroportos — Projeção PCA 2D (k={K_FINAL})")
ax.legend(loc="upper right", framealpha=0.8)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# Loadings — o que cada PC captura?
loadings = pd.DataFrame(
    pca_2d.components_.T,
    index=feature_cols,
    columns=["PC1", "PC2"]
).sort_values("PC1", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for i, pc in enumerate(["PC1", "PC2"]):
    data = loadings[pc].sort_values()
    colors_bar = ["#d62728" if v > 0 else "#1f77b4" for v in data]
    axes[i].barh(data.index, data.values, color=colors_bar)
    axes[i].axvline(0, color="black", linewidth=0.8)
    axes[i].set_title(f"Loadings — {pc} ({pca_2d.explained_variance_ratio_[i]*100:.1f}%)")
    axes[i].set_xlabel("Loading")
    axes[i].grid(alpha=0.25, axis="x")

plt.tight_layout()
plt.show()

## 6. Silhouette plot por cluster

In [ ]:
sample_silhouette_values = silhouette_samples(X_scaled, ap_pd["cluster"])
avg_score = silhouette_score(X_scaled, ap_pd["cluster"])

fig, ax = plt.subplots(figsize=(9, max(4, K_FINAL * 1.2)))
y_lower = 10

for c in range(K_FINAL):
    cluster_vals = np.sort(sample_silhouette_values[ap_pd["cluster"] == c])
    size = cluster_vals.shape[0]
    y_upper = y_lower + size
    color = cm.nipy_spectral(float(c) / K_FINAL)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_vals, alpha=0.7, color=color)
    ax.text(-0.05, y_lower + 0.5 * size, str(c), fontsize=10)
    y_lower = y_upper + 10

ax.axvline(x=avg_score, color="red", linestyle="--", label=f"Avg silhouette = {avg_score:.3f}")
ax.set_xlabel("Silhouette coefficient")
ax.set_ylabel("Cluster")
ax.set_title(f"Silhouette plot — k={K_FINAL}")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 7. Conclusões e interpretação

*(Preencha após rodar o notebook com seus dados reais)*

### O que cada cluster representa?

Analise o heatmap dos centróides e o perfil médio (seção 4) para nomear cada cluster.  
Exemplos comuns nesse tipo de dataset:

| Cluster | Rótulo sugerido | Características típicas |
|---------|----------------|-------------------------|
| 0 | Hubs problemáticos | Alta taxa de atraso, muitos destinos, alta dispersão |
| 1 | Aeroportos regionais tranquilos | Poucos voos, baixa taxa de atraso, distâncias curtas |
| 2 | Hubs eficientes | Alto volume, taxa de atraso controlada |
| ... | ... | ... |

### Limitações
- KMeans assume clusters esféricos — aeroportos com perfis muito irregulares podem ser mal alocados
- Usamos apenas o aeroporto de **origem**; incluir destino poderia revelar padrões de rota
- Dados de um único ano podem não capturar sazonalidade interanual

### Próximos passos
- Testar DBSCAN para detectar aeroportos anomalos (outliers)
- Incluir variáveis geográficas (latitude/longitude, região do país)
- Cruzar os clusters com os resultados da classificação (quais clusters têm maior erro de predição?)